# Velutina — Detector YOLO manos libres en Kaggle (sin kaggle.json)

Cuaderno **100% manos libres**: una vez le das a *Run*, sigue entrenando en los servidores de Kaggle aunque apagues tu PC. No necesitas `kaggle.json` ni descargar nada en tu ordenador.

- **Dataset**: Hornet3000 (Kaggle) `vespa-velutina-v-crabro-vespulina-vulgaris` — ya vive dentro de Kaggle, lo anadimos como *Input* (no se descarga).
- **Que hace**: entrena un **detector** YOLO que dibuja cajas (no un clasificador que solo dice "que bicho es").
- **Idempotente**: puedes darle a *Run All* varias veces sin romper nada. Cada celda imprime que hace.
- **Tiempo**: ~1h con GPU gratis. Cuota Kaggle: 30h/semana, 9h por sesion.


## Celda 1 — GPU y ajustes (config arriba del todo)

Comprueba si hay GPU y deja 4 variables claras arriba. Todo el cuaderno usa esas 4.


In [ ]:
# --- Celda 1: GPU + instalacion + configuracion ---
import torch
print("PyTorch:", torch.__version__)
print("Hay GPU:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Nombre GPU:", torch.cuda.get_device_name(0))
    print("GPUs disponibles:", torch.cuda.device_count())
else:
    print("Sin GPU: el entreno usara CPU (muy lento). Activa GPU en Settings -> Accelerator -> GPU T4 x2.")

# Instala YOLO (ultralytics) — idempotente, si ya esta no hace nada malo
print("Instalando ultralytics...")
get_ipython().system('pip install -q ultralytics')
print("ultralytics listo.")

# ===== CONFIGURACION (toca solo aqui si quieres cambiar algo) =====
MODEL="yolov8n.pt"
EPOCHS=50
IMGSZ=640
BATCH=16
print(f"Config: MODEL={MODEL} EPOCHS={EPOCHS} IMGSZ={IMGSZ} BATCH={BATCH}")
print(" hornet3000 — dataset Hornet3000 / vespa-velutina-v-crabro-vespulina-vulgaris")


## Celda 2 — Localiza el dataset Hornet3000 dentro de Kaggle Input

En Kaggle el dataset NO se descarga: lo anades como *Input* y aparece en `/kaggle/input/vespa-velutina-v-crabro-vespulina-vulgaris/`.
Esta celda lo busca y imprime un arbol de 2 niveles para debug. Si no encuentra `config.yaml`/`data.yaml`, busca `train/images` y avisa que construiremos el yaml a mano.


In [ ]:
# --- Celda 2: localizar dataset Hornet3000 en /kaggle/input ---
from pathlib import Path
import glob as _glob
import os

INPUT_ROOT = Path("/kaggle/input")
print(f"INPUT_ROOT={INPUT_ROOT} existe={INPUT_ROOT.exists()}")
if INPUT_ROOT.exists():
    print("Contenido de /kaggle/input (nivel 1):")
    for p in sorted(INPUT_ROOT.iterdir()):
        print(f"  - {p.name} ({'DIR' if p.is_dir() else 'FILE'})")
        if p.is_dir():
            try:
                subs = sorted(p.iterdir())
                for s in subs[:20]:
                    print(f"      {s.name} ({'DIR' if s.is_dir() else 'FILE'})")
                if len(subs) > 20:
                    print(f"      ... y {len(subs)-20} mas")
            except Exception as e:
                print(f"      (no se pudo listar: {e})")
else:
    print("ERROR: no existe /kaggle/input - has anadido el dataset en Input? Ver docs/KAGGLE.md")

# Busca config.yaml / data.yaml por glob recursivo (hornet3000 suele traer config.yaml)
cands = []
for pat in ["**/config.yaml", "**/data.yaml"]:
    found = list(INPUT_ROOT.rglob(pat)) if INPUT_ROOT.exists() else []
    print(f"Glob {pat}: {len(found)} encontrados")
    for f in found[:10]:
        print(f"  {f}")
    cands.extend(found)

YAML_ORIG = None
for c in cands:
    try:
        t = c.read_text(encoding="utf-8", errors="ignore")
        if "train" in t and ("names" in t or "nc" in t):
            YAML_ORIG = c
            print(f">>> YAML elegido: {YAML_ORIG}")
            print(t[:1200])
            break
    except Exception as e:
        print(f"No se pudo leer {c}: {e}")

if YAML_ORIG is None:
    print("No se encontro YAML valido con train/names. Buscando train/images como fallback...")
    train_imgs = list(INPUT_ROOT.rglob("train/images")) if INPUT_ROOT.exists() else []
    print(f"  train/images encontrados: {len(train_imgs)}")
    for p in train_imgs[:5]:
        print(f"    {p}")
        try:
            subs = [d.name for d in p.iterdir() if d.is_dir()]
            print(f"      subcarpetas: {subs[:5]}")
        except:
            pass
    YAML_ORIG = None
    print("Se construira /kaggle/working/data.yaml manualmente en la siguiente celda.")
else:
    print(f"YAML_ORIG listo: {YAML_ORIG}")
print("Dataset hornet3000 / vespa-velutina-v-crabro-vespulina-vulgaris localizado (si ves rutas arriba, todo bien).")


## Celda 3 — Prepara data.yaml plano en /kaggle/working

El dataset Hornet3000 trae a veces `train/images/Vespa_velutina/...` con subcarpetas. YOLO quiere todo plano: `train/images/*.jpg` y `train/labels/*.txt` sin subcarpetas.
Esta celda copia (aplana) a `/kaggle/working/det/{train,val}/{images,labels}/` igual que hace `sim/dataset/prepare_hornet.py` en local, y genera `/kaggle/working/data.yaml` con `path: /kaggle/working/det`, `train: train/images`, `val: val/images`, `names: [vespa_velutina, vespa_crabro, vespula_vulgaris]` y `nc: 3`.
Idempotente: si ya existe y tiene mismo tamano, no recopia.


In [ ]:
# --- Celda 3: aplanar dataset a /kaggle/working/det y generar data.yaml ---
from pathlib import Path
import shutil, os
import glob

WORK = Path("/kaggle/working")
DET = WORK / "det"
DATA_YAML = WORK / "data.yaml"
INPUT_ROOT = Path("/kaggle/input")

NAMES = ["vespa_velutina", "vespa_crabro", "vespula_vulgaris"]
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

print(f"WORK={WORK} DET={DET} DATA_YAML={DATA_YAML}")
DET.mkdir(parents=True, exist_ok=True)

try:
    YAML_ORIG
except NameError:
    YAML_ORIG = None

if YAML_ORIG and YAML_ORIG.exists():
    base = YAML_ORIG.parent
    for up in [base] + list(base.parents)[:4]:
        cand = up / "train" / "images"
        if cand.exists():
            base = up
            break
    print(f"Base deducida desde YAML: {base}")
    SRC_TRAIN_IMAGES = base / "train" / "images"
    SRC_TRAIN_LABELS = base / "train" / "labels"
    SRC_VAL_IMAGES = base / "val" / "images"
    SRC_VAL_LABELS = base / "val" / "labels"
    if not SRC_VAL_IMAGES.exists():
        alt = base / "valid" / "images"
        if alt.exists():
            SRC_VAL_IMAGES = alt
            SRC_VAL_LABELS = base / "valid" / "labels"
            print(f"Usando valid/ en vez de val/: {alt}")
else:
    cands = list(INPUT_ROOT.rglob("train/images")) if INPUT_ROOT.exists() else []
    if cands:
        SRC_TRAIN_IMAGES = cands[0]
        base = SRC_TRAIN_IMAGES.parent.parent
        SRC_TRAIN_LABELS = base / "train" / "labels"
        SRC_VAL_IMAGES = base / "val" / "images"
        SRC_VAL_LABELS = base / "val" / "labels"
        if not SRC_VAL_IMAGES.exists():
            alt = base / "valid" / "images"
            if alt.exists():
                SRC_VAL_IMAGES = alt
                SRC_VAL_LABELS = base / "valid" / "labels"
        print(f"Base deducida por glob: {base}")
    else:
        print("ERROR: no se encontro train/images en /kaggle/input")
        print("Asegurate de haber hecho Add data -> vespa-velutina-v-crabro-vespulina-vulgaris")
        raise SystemExit("No se encontro dataset Hornet3000 en /kaggle/input")

for p in [SRC_TRAIN_IMAGES, SRC_TRAIN_LABELS, SRC_VAL_IMAGES, SRC_VAL_LABELS]:
    print(f"  {p} existe={p.exists() if p else False}")

def find_images(src_dir: Path):
    out=[]
    if not src_dir or not src_dir.exists():
        return out
    for f in src_dir.rglob("*"):
        if f.is_file() and f.suffix.lower() in IMAGE_EXTS:
            out.append(f)
    return sorted(out)

def find_label_for_image(img_path: Path, src_labels_dir: Path):
    stem = img_path.stem
    try:
        rel = img_path.relative_to(src_labels_dir.parent / "images")
        cand = src_labels_dir / rel.with_suffix(".txt")
        if cand.exists():
            return cand
    except Exception:
        pass
    if src_labels_dir and src_labels_dir.exists():
        matches = list(src_labels_dir.rglob(f"{stem}.txt"))
        if matches:
            return matches[0]
    return None

def flatten_copy(src_img_dir, src_lbl_dir, dst_img_dir, dst_lbl_dir):
    dst_img_dir.mkdir(parents=True, exist_ok=True)
    dst_lbl_dir.mkdir(parents=True, exist_ok=True)
    imgs = find_images(src_img_dir)
    print(f"  {src_img_dir} -> {dst_img_dir}: {len(imgs)} imagenes encontradas")
    subs = [d.name for d in src_img_dir.iterdir() if d.is_dir()] if src_img_dir.exists() else []
    if subs:
        print(f"    subcarpetas detectadas: {subs}")
    copied_img = copied_lbl = bg = 0
    for img in imgs:
        dst_img = dst_img_dir / img.name
        do_copy = True
        if dst_img.exists() and dst_img.stat().st_size == img.stat().st_size:
            do_copy = False
        if do_copy:
            shutil.copy2(img, dst_img)
            copied_img += 1
        lbl = find_label_for_image(img, src_lbl_dir) if src_lbl_dir else None
        if lbl and lbl.exists():
            dst_lbl = dst_lbl_dir / f"{img.stem}.txt"
            do_copy_lbl = True
            if dst_lbl.exists() and dst_lbl.stat().st_size == lbl.stat().st_size:
                do_copy_lbl = False
            if do_copy_lbl:
                shutil.copy2(lbl, dst_lbl)
                copied_lbl += 1
        else:
            bg += 1
    print(f"    copiadas nuevas: imgs={copied_img} labels={copied_lbl} fondo_sin_label={bg}")
    return len(imgs)

print("Aplanando train...")
n_train = flatten_copy(SRC_TRAIN_IMAGES, SRC_TRAIN_LABELS, DET / "train" / "images", DET / "train" / "labels")
print("Aplanando val...")
n_val = flatten_copy(SRC_VAL_IMAGES, SRC_VAL_LABELS, DET / "val" / "images", DET / "val" / "labels")

for split in ("train","val"):
    imgs = [p for p in (DET / split / "images").glob("*.*") if p.suffix.lower() in IMAGE_EXTS] if (DET / split / "images").exists() else []
    lbls = list((DET / split / "labels").glob("*.txt")) if (DET / split / "labels").exists() else []
    print(f"[verify] {split} plano: images={len(imgs)} labels={len(lbls)}")

DATA_YAML.write_text(
    f"path: {DET.as_posix()}
"
    f"train: train/images
"
    f"val: val/images
"
    f"nc: {len(NAMES)}
"
    f"names: [{', '.join(NAMES)}]
",
    encoding="utf-8"
)
print(f"Generado {DATA_YAML}:")
print(DATA_YAML.read_text(encoding="utf-8"))
print("Listo: /kaggle/working/data.yaml apunta a /kaggle/working/det (train/images, val/images).")


## Celda 4 — Entrena el detector YOLO

Entrena `YOLO(MODEL).train(data=/kaggle/working/data.yaml, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=0, project=/kaggle/working/runs, name=det)`.
Sin `kaggle.json`, sin descargas: todo sale de `/kaggle/input` y escribe en `/kaggle/working`.
Tarda ~1h con GPU T4 x2. Puedes apagar el PC tranquilo: Kaggle sigue entrenando en su servidor.


In [ ]:
# --- Celda 4: entrena YOLO ---
from ultralytics import YOLO
from pathlib import Path

DATA_YAML = Path("/kaggle/working/data.yaml")
print(f"DATA_YAML={DATA_YAML} existe={DATA_YAML.exists()}")
print(DATA_YAML.read_text(encoding="utf-8") if DATA_YAML.exists() else "FALTA data.yaml")

try:
    MODEL
except NameError:
    MODEL="yolov8n.pt"; EPOCHS=50; IMGSZ=640; BATCH=16

print(f"Entrenando {MODEL} con data={DATA_YAML} epochs={EPOCHS} imgsz={IMGSZ} batch={BATCH} device=0 project=/kaggle/working/runs name=det")
model = YOLO(MODEL)
model.train(data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=0, project="/kaggle/working/runs", name="det")
print("Entrenamiento terminado.")
BEST = Path("/kaggle/working/runs/det/weights/best.pt")
print(f"best.pt existe={BEST.exists()} tam={BEST.stat().st_size/1e6:.1f} MB" if BEST.exists() else "best.pt NO encontrado (revisa runs/)")
for p in sorted(Path("/kaggle/working/runs").rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to('/kaggle/working')} {p.stat().st_size/1e6:.2f} MB")


## Celda 5 — Evalua: mAP y recall velutina + FP duros

Usa `model.val()` para mAP50, mAP50-95 y precision/recall por clase. Ademas recorre val con `model.predict` a conf 0.5 para medir **recall velutina** (de cada 100 velutinas, cuantas pilla) y **FP duros** (cajas velutina en imagenes sin velutina). Guarda `/kaggle/working/resultados_det_kaggle.txt` y `/kaggle/working/box_thresholds_kaggle.json`.


In [ ]:
# --- Celda 5: model.val() + recall velutina + FP duros; guarda txt/json ---
from ultralytics import YOLO
from pathlib import Path
import json
from collections import Counter

BEST = Path("/kaggle/working/runs/det/weights/best.pt")
DATA_YAML = Path("/kaggle/working/data.yaml")
print(f"BEST={BEST} existe={BEST.exists()}")
print(f"DATA_YAML={DATA_YAML} existe={DATA_YAML.exists()}")

model = YOLO(str(BEST) if BEST.exists() else MODEL)
metrics = model.val(data=str(DATA_YAML))
print("--- Metricas globales (model.val) ---")
print(f"mAP50:    {float(metrics.box.map50):.4f}")
print(f"mAP50-95: {float(metrics.box.map):.4f}")
print(f"precision:{float(metrics.box.mp):.4f}  recall:{float(metrics.box.mr):.4f}")
print("--- Por clase (0=vespa_velutina, 1=vespa_crabro, 2=vespula_vulgaris) ---")
try:
    import numpy as np
    names = metrics.names
    ap50 = np.array(metrics.box.ap50)
    ap = np.array(metrics.box.ap)
    p = np.array(metrics.box.p)
    r = np.array(metrics.box.r)
    for i, n in names.items():
        print(f"  clase {i} ({n}): AP50={ap50[i]:.4f} AP50-95={ap[i]:.4f} P={p[i]:.4f} R={r[i]:.4f}")
except Exception as e:
    print(f"(detalle por clase no disponible: {e})")

CONF = 0.5
VEL = 0
DET = Path("/kaggle/working/det")
val_imgs = sorted([p for p in (DET / "val" / "images").glob("*.*") if p.suffix.lower() in {".jpg",".jpeg",".png",".bmp",".webp"}])
print(f"Imagenes val a revisar: {len(val_imgs)} (desde /kaggle/working/det/val/images)")

def label_has_velutina(img_path: Path):
    cand = (DET / "val" / "labels" / (img_path.stem + ".txt"))
    if not cand.exists():
        cand2 = Path(str(img_path).replace("images","labels")).with_suffix(".txt")
        cand = cand2 if cand2.exists() else cand
    if not cand.exists():
        return None
    for ln in cand.read_text(encoding="utf-8", errors="ignore").splitlines():
        parts = ln.strip().split()
        if not parts:
            continue
        try:
            if int(float(parts[0])) == VEL:
                return True
        except:
            pass
    return False

n_vel_imgs = n_vel_ok = n_fp_duros = n_sin = 0
fp_origen = Counter()
for img in val_imgs:
    truth = label_has_velutina(img)
    if truth is None:
        n_sin += 1
        continue
    res = model.predict(str(img), conf=CONF, verbose=False)[0]
    pred_vel = any(int(c) == VEL for c in res.boxes.cls.tolist()) if len(res.boxes) else False
    if truth:
        n_vel_imgs += 1
        if pred_vel:
            n_vel_ok += 1
    else:
        if pred_vel:
            n_fp_duros += 1
            cand = DET / "val" / "labels" / (img.stem + ".txt")
            t = cand.read_text(encoding="utf-8", errors="ignore") if cand.exists() else ""
            cls = sorted({l.split()[0] for l in t.splitlines() if l.strip().split()}) if t.strip() else []
            fp_origen[",".join(cls) if cls else "fondo/sin_caja"] += 1

recall_vel = (n_vel_ok / n_vel_imgs) if n_vel_imgs else 0.0
print(f"--- Resultado propio (conf={CONF}) ---")
print(f"Imagenes val con velutina: {n_vel_imgs}; detectadas: {n_vel_ok} -> recall velutina@{CONF}={recall_vel:.3f}")
print(f"Cajas velutina en imgs SIN velutina (FP duros): {n_fp_duros}")
print(f"Origen FP duros: {dict(fp_origen)}")
print(f"(sin etiqueta, ignoradas: {n_sin})")

out_txt = Path("/kaggle/working/resultados_det_kaggle.txt")
out_txt.write_text(
    f"mAP50={float(metrics.box.map50):.4f} mAP50-95={float(metrics.box.map):.4f} P={float(metrics.box.mp):.4f} R={float(metrics.box.mr):.4f}
"
    f"recall_velutina@{CONF}={recall_vel:.4f} ({n_vel_ok}/{n_vel_imgs})
"
    f"FP_duros_velutina_en_no_velutina={n_fp_duros} origen={dict(fp_origen)}
"
    f"conf={CONF} val_imgs={len(val_imgs)} modelo={MODEL} epochs={EPOCHS} imgsz={IMGSZ} batch={BATCH}
"
    f"best.pt={BEST}
",
    encoding="utf-8"
)
print(f"Guardado {out_txt} ({out_txt.stat().st_size} bytes):")
print(out_txt.read_text(encoding="utf-8"))

thr = {
    "conf_default": CONF,
    "recall_velutina": round(float(recall_vel),4),
    "fp_duros": int(n_fp_duros),
    "fp_origen": dict(fp_origen),
    "mAP50": round(float(metrics.box.map50),4),
    "mAP50_95": round(float(metrics.box.map),4),
    "precision": round(float(metrics.box.mp),4),
    "recall": round(float(metrics.box.mr),4),
    "val_imgs": len(val_imgs),
    "n_vel_imgs": int(n_vel_imgs),
    "modelo": MODEL,
    "epochs": int(EPOCHS),
    "imgsz": int(IMGSZ),
    "batch": int(BATCH),
}
out_json = Path("/kaggle/working/box_thresholds_kaggle.json")
out_json.write_text(json.dumps(thr, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Guardado {out_json}:")
print(out_json.read_text(encoding="utf-8"))


## Celda 6 — Donde descargar (Output de Kaggle)

En Kaggle, panel derecho **Output** veras todo lo que queda en `/kaggle/working` al terminar.

- `best.pt` estara en `/kaggle/working/runs/det/weights/best.pt` (~6 MB)
- `resultados_det_kaggle.txt` y `box_thresholds_kaggle.json` estaran en `/kaggle/working/`

Instruccion al volver a tu repo local:
1. En Kaggle, Output -> click en `best.pt` -> Download (o Download All).
2. En tu repo guarda ese fichero como `jetson/ai/yolov8n_det_kaggle.pt` (nombre distinto a `yolov8n_det.pt` y a los cls).
3. Guarda tambien `resultados_det_kaggle.txt` y `box_thresholds_kaggle.json` junto al peso para comparar.
No necesitas descargar el dataset ni los logs: solo esos 3 ficheros (~6 MB + 2 txt/json pequenos).


In [ ]:
# --- Celda 6: muestra rutas de descarga ---
from pathlib import Path
import os

BEST = Path("/kaggle/working/runs/det/weights/best.pt")
TXT = Path("/kaggle/working/resultados_det_kaggle.txt")
JS = Path("/kaggle/working/box_thresholds_kaggle.json")

print("Ficheros a descargar desde Kaggle Output:")
for p in [BEST, TXT, JS]:
    if p.exists():
        print(f"  {p} -> {p.stat().st_size/1e6:.2f} MB / {p.stat().st_size} bytes")
    else:
        print(f"  {p} NO existe aun (has ejecutado celda 5?)")

print("")
print("En Kaggle: panel derecho Output -> veras /kaggle/working. Descarga best.pt (~6MB).")
print("En local guarda como: jetson/ai/yolov8n_det_kaggle.pt")
print("Y guarda tambien resultados_det_kaggle.txt y box_thresholds_kaggle.json junto al peso.")
print("\n--- /kaggle/working (2 niveles) ---")
for p in sorted(Path("/kaggle/working").rglob("*")):
    if p.is_file():
        rel = p.relative_to("/kaggle/working")
        if len(rel.parts) <= 3:
            print(f"  {rel} {p.stat().st_size/1e6:.2f} MB")


## Lectura honesta: detector vs clasificador, 50 epocas y que falta

**Detector vs clasificador.**
- El **clasificador** (`yolov8n-cls`) responde a una foto recortada: *que bicho es este?* Solo clasifica.
- El **detector** (este cuaderno, `yolov8n.pt`) responde a una escena entera: *donde hay bichos y de que tipo es cada uno?* Dibuja cajas + clase. Para el apiario el detector es lo que vale: localiza en vuelo.

**50 epocas.**
Con `EPOCHS=50`, `IMGSZ=640`, `BATCH=16` y `yolov8n.pt` el entreno tarda ~1h en T4 x2 y suele dar mAP50 decente sin sobreajustar mucho. Con menos epocas (p. ej. 30) el modelo queda corto; con muchas mas (100+) el val pequeno puede sobreajustar y empeorar en campo. Si tu val es pequeno, 50 es un buen compromiso; si ves overfitting (train sube, val baja), baja a 30-40 o anade augment.

**Limites (importante).**
- Fotos de **observadores != vuelo en tu apiario**: el dataset Hornet3000 son fotos grandes, centradas, con luz buena y bicho quieto. En tu piquera el bicho sale pequeno, movido, con tu fondo y tu luz. Los numeros de Kaggle (mAP, recall) NO son numeros en campo.
- **Val pequeno y sesgado**: si val tiene pocas imagenes o pocas escenas de tu apiario, el mAP adorna. No te fies solo del txt.
- **Clases desbalanceadas**: velutina/crabro/vulgaris pueden no estar equilibradas; el modelo puede favorecer la mayoria.
- **Sin track temporal**: una sola caja suelta no basta para decidir disparo.

**Que falta para disparo (no disparar solo con este cuaderno).**
1. **Validacion en escena real**: graba tu piquera/fondo con tu camara, anota unas decenas de fotos y mide recall/FP ahi (el `resultados_det_kaggle.txt` de Kaggle no vale para campo).
2. **Calibra umbral**: sube `conf` hasta que los FP duros (crabro/vulgaris/fondo) sean ~0 y mira cuantas velutinas sigues pillando. Usa `box_thresholds_kaggle.json` como punto de partida, pero repite la curva en tu val real.
3. **Regla de disparo**: exige varias detecciones seguidas (track/temporales) antes de actuar; una sola caja a 0.5 no es fiable.
4. **Prueba de selectividad y seguridad**: verifica fauna, IEC y failsafe antes de habilitar laser.

Refs: dataset https://www.kaggle.com/datasets/marcoryvandijk/vespa-velutina-v-crabro-vespulina-vulgaris y guia https://github.com/vespCV/hornet3000
